In [ ]:
import pandas as pd
import os

from google.colab import files

# Téléchargez à nouveau le fichier
print("Veuillez télécharger le fichier:")
uploaded = files.upload()



Veuillez télécharger le fichier:


Saving merged_data.xlsx to merged_data.xlsx


In [ ]:
import pandas as pd

cv = pd.read_excel('merged_data.xlsx')
cv_original = cv.copy()  # Garder une copie de sauvegarde
cv = cv.copy()
colonnes_a_supprimer = ['constructible', 'anneeConst','delegation','installations_sportives','localite','plein_air','_source_file','service','_source_sheet','Column1','Column2','Column3','Column4','Column5','Column6','Column7','Column8','Column9','Column10','Column11','Column12','Column13','Column14','Column15','Column16','Column17','Column18','Column19','Column20','Column21','Column22','Column23','Column24','Column25','Column26','Column27']  # Remplacez par vos colonnes
cv = cv.drop(columns=colonnes_a_supprimer)

In [ ]:
print(f"\nDonnées nettoyées: {cv.shape}")


Données nettoyées: (114961, 20)


In [ ]:
cv = cv.drop_duplicates()


In [ ]:
print(f"\nDonnées nettoyées: {cv.shape}")
print(f"Colonnes restantes: {list(cv.columns)}")


Données nettoyées: (77851, 20)
Colonnes restantes: ['adresse', 'agence', 'chauffage', 'climatisation', 'codeP', 'cuisine', 'dateAnnonce', 'description', 'fonds', 'gouvernorat', 'link', 'nbpiece', 'other_data', 'price', 'reference', 'salle_de_bain', 'superficie_habitable', 'superficie_terrain', 'tel', 'typeImm']


In [ ]:
cv = cv.dropna(subset=['description'])


In [ ]:
print(f"Forme finale: {cv.shape}")
print(f"Valeurs manquantes dans description: {cv['description'].isnull().sum()}")
print(cv.head())

Forme finale: (77826, 20)
Valeurs manquantes dans description: 0
    adresse              agence chauffage climatisation  codeP cuisine  \
0  L'AOUINA      dammak sabrine       NaN           NaN    NaN     NaN   
1  HAMMAMET  Agence SMART OWNER       NaN           NaN    NaN     NaN   
2  ENNASR 2     Building Négoce       NaN           NaN    NaN     NaN   
3  gammarth        cherif hadil       NaN           NaN    NaN     NaN   
4  l'aouina      dammak sabrine       NaN           NaN    NaN     NaN   

  dateAnnonce                                        description fonds  \
0       44992     Superficie  : 105 m²  Prix :  270 MD  REF :...   NaN   
1       44992                                                      Oui   
2       44992                                                ...   NaN   
3       44992     Superficie  : 110 m²  Prix :  350MD  REF : ...   NaN   
4       44992    ✔ Superficie : 150 m² ✔ Prix : 360 MD ✔ REF ...   NaN   

  gouvernorat                                

In [ ]:
cv['typeImm'] = cv['typeImm'].str.split('-').str[-1].str.strip()

# Voir le résultat
print("=== APRÈS NETTOYAGE ===")
print(cv['typeImm'].head(10))
print(f"\nValeurs uniques: {cv['typeImm'].unique()}")

In [ ]:
def regrouper_type_bien(texte):
    if pd.isna(texte):
        return 'Non spécifié'

    texte = str(texte).lower().strip()


    if any(word in texte for word in ['appartement', 'appartements', 'appart', 'duplex', 'triplex', 'studio']):
        return 'Appartement'

    # Villas et maisons
    if any(word in texte for word in ['villa', 'maison', 'rdc de villa', 'etage de villa', 'maison / villa']):
        return 'Villa/Maison'

    # Terrains (tous types)
    if any(word in texte for word in ['terrain', 'terrains', 'agricole', 'constructible', 'promoteur', 'commercial', 'loisir']):
        return 'Terrain'

    if 'ferme' in texte:
        return 'Ferme'

    if 'immeuble' in texte:
        return 'Immeuble'

    if any(word in texte for word in ['local commercial', 'bureau', 'fond de commerce', 'dépot', 'depot']):
        return 'Local/Bureau'

    if 'usine' in texte:
        return 'Usine'
    if 'hotel' in texte:
        return 'Hôtel'

    return 'Autre'

cv['typeImm_regroupe'] = cv['typeImm'].apply(regrouper_type_bien)

print("=== COMPARAISON AVANT/APRÈS ===")
comparaison = cv[['typeImm', 'typeImm_regroupe']].drop_duplicates()
print(comparaison.to_string(max_rows=50))

print("\n=== DISTRIBUTION APRÈS REGROUPEMENT ===")
print(cv['typeImm_regroupe'].value_counts())

In [ ]:
colonnes_a_supprimer = ['typeImm']  # Remplacez par vos colonnes
cv = cv.drop(columns=colonnes_a_supprimer)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
valeurs_manquantes = cv.isnull().sum()
valeurs_manquantes = valeurs_manquantes[valeurs_manquantes > 0]
total_lignes = len(cv)

# Créer le tableau
tableau_manquantes = pd.DataFrame({
    'Colonne': valeurs_manquantes.index,
    'Nb valeurs manquantes': valeurs_manquantes.values,
    'Pourcentage (%)': (valeurs_manquantes.values / total_lignes * 100).round(2)
})
tableau_manquantes = tableau_manquantes.sort_values('Nb valeurs manquantes', ascending=False)

# Afficher le tableau avec style (pour notebook)
from IPython.display import display

print("TABLEAU DES VALEURS MANQUANTES")

display(tableau_manquantes.style.background_gradient(cmap='Reds', subset=['Nb valeurs manquantes', 'Pourcentage (%)']))

# Graphique
plt.figure(figsize=(12, 6))
plt.bar(valeurs_manquantes.index, valeurs_manquantes.values, color='coral', edgecolor='darkred', alpha=0.7)
plt.xlabel('Colonnes', fontsize=12)
plt.ylabel('Nombre de valeurs manquantes', fontsize=12)
plt.title('Nombre de valeurs manquantes par colonne', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import unicodedata
import html
import re

class DescriptionCleaner:
    def __init__(self):
        self.currency_mapping = {
            'dt': 'DT', 'dinar': 'DT', 'tnd': 'DT', 'دينار': 'DT',
            'euro': '€', 'usd': '$', 'dinars': 'DT'
        }
        self.error_patterns = [
            (r'(.)\1{3,}', r'\1\1'),  # Répétitions excessives
            (r'\s+', ' '),             # Espaces multiples
            (r'[^\w\s\.\,\!\?\-\'\"\/\€\$\u00C0-\u00FF\u0600-\u06FF0-9]', ' ')
        ]

    def normalize_unicode(self, text: str) -> str:
        """Normaliser les caractères Unicode"""
        if pd.isna(text):
            return text
        text = unicodedata.normalize('NFKD', str(text))
        text = html.unescape(text)
        return text



    def clean_currency(self, text: str) -> str:
        """Normaliser les devises (corrigé pour '500dt')"""
        if pd.isna(text):
            return text

        text_lower = text.lower()
        for currency, symbol in self.currency_mapping.items():
            if currency in text_lower:
                # Cas 1: "500dt" ou "500 dt" -> "500 DT"
                text = re.sub(r'(\d+)\s*' + currency + r'\b', r'\1 ' + symbol, text, flags=re.IGNORECASE)
                # Cas 2: "dt500" -> "DT 500"
                text = re.sub(r'\b' + currency + r'\s*(\d+)', r'' + symbol + r' \1', text, flags=re.IGNORECASE)
                # Cas 3: "dt" seul
                text = re.sub(r'\b' + currency + r'\b', symbol, text, flags=re.IGNORECASE)

        return text

    def clean_whitespace(self, text: str) -> str:
        """Nettoyer les espaces et répétitions"""
        if pd.isna(text):
            return text
        for pattern, replacement in self.error_patterns:
            text = re.sub(pattern, replacement, text)
        return text.strip()

    def clean_text(self, text: str) -> str:
        """Pipeline complet de nettoyage"""
        if pd.isna(text):
            return ''

        text = self.normalize_unicode(text)
        text = self.clean_currency(text)  # Version corrigée
        text = self.clean_whitespace(text)

        return text

# Créer une instance du cleaner
cleaner = DescriptionCleaner()

# Tester avec des exemples
test_texts = [
    "Appartement 120m² prix 500dt",
    "Villa à vendre 800000 dt",
    "Terrain dt 250000",
    "Bureau 1500dt par mois",
    "Maison 300000 dinars"
]

print("=== TEST DE CONVERSION DES DEVISES ===\n")
for test in test_texts:
    result = cleaner.clean_currency(test)
    print(f"Avant: {test}")
    print(f"Après: {result}")
    print("-" * 50)

# Appliquer à votre dataframe
print("\n=== APPLICATION À VOTRE DATAFRAME ===")
cv['description_clean'] = cv['description'].apply(cleaner.clean_text)

In [ ]:
import pandas as pd
import unicodedata
import html

# Définir la fonction
def normalize_unicode(self, text: str) -> str:
    text = unicodedata.normalize('NFKD', text)
    text = html.unescape(text)
    return text

# Créer une instance (si dans une classe) ou utiliser directement
def normalize_unicode_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', str(text))
    text = html.unescape(text)
    return text

# Afficher AVANT normalisation
print("="*80)
print("AVANT NORMALISATION UNICODE")
print("="*80)
print(cv['description'].head(10).to_string())
print("\n" + "="*80)

# Appliquer la normalisation
cv['description_normalized'] = cv['description'].apply(normalize_unicode_text)

# Afficher APRÈS normalisation
print("\n" + "="*80)
print("APRÈS NORMALISATION UNICODE")
print("="*80)
print(cv['description_normalized'].head(10).to_string())
print("\n" + "="*80)

# Comparaison côte à côte
print("\n" + "="*80)
print("COMPARAISON CÔTE À CÔTE (5 premières lignes)")
print("="*80)
for i in range(min(5, len(cv))):
    print(f"\n--- LIGNE {i+1} ---")
    print(f"Avant: {cv['description'].iloc[i]}")
    print(f"Après: {cv['description_normalized'].iloc[i]}")
    print("-"*50)

# Remplacer l'ancienne colonne par la nouvelle
cv['description'] = cv['description_normalized']
cv = cv.drop(columns=['description_normalized'])


In [ ]:
cv.to_excel('cv_nettoye.xlsx', index=False)
